In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Iiführig in de Qiskit KI-gstützte Transpiler-Service

*Gschätzte QPU-Nutzung: Kei (HINWEIS: Des Tutorial führt kei Jobs us, wil's sich uf Transpilation fokussiert)*

## Hintergrund
Dr **Qiskit KI-gstützte Transpiler-Service (QTS)** führt maschinelles-Lernen-basierti Optimierunge sowohl in Routing- als au in Synthesis-Passes ii. Die KI-Modi sin entwickelt worre, um d' Beschränkunge von dr traditionelle Transpilation azgehe, insbesondere für grossi Schaltunge un komplexe Hardware-Topologie.

Ab **Juli 2025** isch dr **Transpiler-Service** zur neue IBM Quantum&reg; Platform migriert worre un isch nit mehr verfügbar. Für d' neuschte Updates zum Status vom Transpiler-Service verwise mir uf d' [Transpiler-Service-Dokumentation](/guides/qiskit-transpiler-service). Ir chöit de KI-Transpiler witerhin lokal verwende, ähnlich wie bi dr Standard-Qiskit-Transpilation. Ersetzt eifach `generate_preset_pass_manager()` durch `generate_ai_pass_manager()`. Die Funktion konstruiert e Pass-Manager, wo d' KI-gstützte Routing- un Synthesis-Passes direkt in eire lokale Transpilations-Workflow integriert.

### Hauptmerkmal von de KI-Passes
- Routing-Passes: KI-gstützts Routing ka Qubit-Pfad dynamisch basierend uf d' spezifischi Schaltung un s' Backend aapasst un de Bedarf aa übermässige SWAP-Gates reduziert.
    - `AIRouting`: Layout-Auswahl un Schaltungs-Routing

- Synthesis-Passes: KI-Technike optimiere d' Zerlegung von Multi-Qubit-Gates un minimiere d' Anzahl von de Zwei-Qubit-Gates, wo typischerwyse anfälliger für Fehler sin.
    - `AICliffordSynthesis`: Clifford-Gate-Synthese
    - `AILinearFunctionSynthesis`: Synthese von Linear-Funktionsschaltunge
    - `AIPermutationSynthesis`: Synthese von Permutationsschaltunge
    - `AIPauliNetworkSynthesis`: Synthese von Pauli-Netzwerkschaltunge (nur im Qiskit Transpiler Service verfügbar, nit in dr lokale Umgebung)

- Vergleich mit traditioneller Transpilation: Dr Standard-Qiskit-Transpiler isch e robustes Werkzeug, wo e breits Spektrum von Quanteschaltunge effektiv handhabe ka. Wenn Schaltunge aber grösser werre oder Hardware-Konfiguratione komplexer werre, chöi KI-Passes zusätzliche Optimierungsgewinn liefere. Durch d' Verwendung von glernde Modelle für Routing un Synthese verfeinert QTS Schaltungslayouts witer un reduziert de Overhead für herausfordernderi oder gross aagelegte Quanteaufgabe.

Des Tutorial evaluiert d' KI-Modi unter Verwendung sowohl von Routing- als au von Synthesis-Passes un vergleicht d' Ergebnis mit traditioneller Transpilation, um hervorzuhebe, wo KI Leistungsvorteile bietet.

Witteri Details zu de verfügbare KI-Passes findet ir in dr [KI-Passes-Dokumentation](/guides/ai-transpiler-passes).

### Warum KI für Quanteschaltungs-Transpilation verwende? {#warum-ai-für-quantenschaltungs-transpilation-verwenden}

Wil Quanteschaltunge in Gröss un Komplexität zunehmend wachse, hän traditionelle Transpilationsmethode Schwierigkeite, Layouts zu optimiere un Gate-Anzahle effizient zu reduziere. Grösseri Schaltunge, insbesondere solchi mit Hunderte von Qubits, stelle erhebliche Herausforderunge aa s' Routing un d' Synthese, wege Gerätebeschränkunge, begrenzter Konnektivität un Qubit-Fehlerraten.

Do bietet d' KI-gstützte Transpilation e potenzielli Lösung. Durch d' Nutzung von maschinelle Lerntechnike ka dr KI-gstützte Transpiler in Qiskit gscheiteri Entscheidunge über Qubit-Routing un Gate-Synthese treffe, was zu besserer Optimierung von gross aagelegte Quanteschaltunge führt.

### Kurzi Benchmarking-Ergebnis
![Graph showing AI transpiler performance against Qiskit](../docs/images/tutorials/ai-transpiler-introduction/ai-transpiler-benchmarks.avif)

In Benchmarking-Tests hän dr KI-Transpiler konsistent flacheri Schaltunge höherer Qualität im Vergleich zum Standard-Qiskit-Transpiler produziert. Für die Tests hän mir d' Standard-Pass-Manager-Strategie von Qiskit verwendet, konfiguriert mit [`generate_preset_passmanager`]. Während die Standardstrategie oft effektiv isch, ka sie bi grössere oder komplexere Schaltunge Schwierigkeite hän. Im Gegensatz dazu hän KI-gstützte Passes e durchschnittlichi Reduzierig von dr Zwei-Qubit-Gate-Anzahl um 24% un e Reduzierig von dr Schaltungstiefe um 36% für grossi Schaltunge (100+ Qubits) bi dr Transpilation uf d' Heavy-Hex-Topologie von IBM Quantum Hardware erreicht. Witteri Informatione zu die Benchmarks findet ir in dem [Blog.](https://www.ibm.com/quantum/blog/qiskit-performance)

Des Tutorial untersucht d' wichtigschte Vorteile von de KI-Passes un wie se sich mit traditionelle Methode vergleiche.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import matplotlib.pyplot as plt
from statistics import mean, stdev
import time
import logging

seed = 42


def transpile_with_metrics(pass_manager, circuit):
    """Transpile a circuit and return the result along with key metrics."""
    start = time.time()
    qc_out = pass_manager.run(circuit)
    elapsed = time.time() - start

    depth_2q = qc_out.depth(lambda x: x.operation.num_qubits == 2)
    gate_count = qc_out.size()

    return qc_out, {
        "depth_2q": depth_2q,
        "gate_count": gate_count,
        "time_s": round(elapsed, 3),
    }


def remap_to_contiguous(tqc):
    """Remap a transpiled circuit to use contiguous qubit indices.

    Transpiled circuits target specific physical qubits (e.g., qubit 45, 67)
    on a large backend. This remaps them to 0, 1, 2, ... so Aer only
    simulates the active qubits."""
    active = sorted(
        {tqc.find_bit(q).index for inst in tqc.data for q in inst.qubits}
    )
    qubit_map = {old: new for new, old in enumerate(active)}
    new_qc = QuantumCircuit(len(active))
    for inst in tqc.data:
        old_indices = [tqc.find_bit(q).index for q in inst.qubits]
        new_qc.append(inst.operation, [qubit_map[i] for i in old_indices])
    return new_qc


def build_mirror_circuit(tqc, simulate=True):
    """Build a mirror circuit: U followed by U-dagger, with measurements.

    The expected output is always |0...0>, so measuring the survival
    probability reveals how much noise each transpilation strategy adds.

    Args:
        tqc: A transpiled circuit.
        simulate: If True (default), remap to contiguous qubits so Aer
            only simulates the active qubits. If False, keep the full
            physical layout for hardware execution."""
    if simulate:
        tqc = remap_to_contiguous(tqc)
    mirror = tqc.compose(tqc.inverse())
    mirror.measure_all()
    return mirror


def print_summary(results):
    """Print a summary of each metric as mean +/- stdev across all circuits,
    along with the mean percentage improvement of AI over Default."""
    metrics = [
        ("Depth 2Q", "Depth 2Q (Default)", "Depth 2Q (AI)"),
        ("Gate Count", "Gate Count (Default)", "Gate Count (AI)"),
        ("Time (s)", "Time (Default)", "Time (AI)"),
    ]
    header = (
        f"{'Metric':<12}{'Default (mean +/- std)':>24}"
        f"{'AI (mean +/- std)':>22}{'AI % improvement':>22}"
    )
    print(header)
    print("-" * len(header))
    for label, col_def, col_ai in metrics:
        defaults = [r[col_def] for r in results]
        ais = [r[col_ai] for r in results]
        pct = [(d - a) / d * 100 for d, a in zip(defaults, ais)]
        default_str = f"{mean(defaults):.1f} +/- {stdev(defaults):.1f}"
        ai_str = f"{mean(ais):.1f} +/- {stdev(ais):.1f}"
        pct_str = f"{mean(pct):+.1f}% +/- {stdev(pct):.1f}%"
        print(f"{label:<12}{default_str:>24}{ai_str:>22}{pct_str:>22}")


def plot_metrics_and_pct(results, title_prefix):
    """Plot metric comparisons and percentage improvement of AI over Default."""
    qubits = [r["Qubits"] for r in results]
    metrics = [
        ("Depth 2Q (Default)", "Depth 2Q (AI)", "Two-Qubit Depth"),
        ("Gate Count (Default)", "Gate Count (AI)", "Gate Count"),
        ("Time (Default)", "Time (AI)", "Transpilation Time"),
    ]

    # Row 1: raw metric comparison
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: Metric Comparison",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        ax.plot(qubits, [r[col_def] for r in results], "o-", label="Default")
        ax.plot(qubits, [r[col_ai] for r in results], "s-", label="AI")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel(label)
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Row 2: percentage improvement
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: % Improvement of AI over Default",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        pct = [(r[col_def] - r[col_ai]) / r[col_def] * 100 for r in results]
        ax.axhline(
            0, color="#1f77b4", linewidth=2, label="Default (baseline)"
        )
        ax.plot(qubits, pct, "s-", color="#ff7f0e", label="AI")
        ax.fill_between(qubits, 0, pct, alpha=0.15, color="#ff7f0e")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel("% Improvement")
        ax.legend()
    plt.tight_layout()
    plt.show()


# Suppress verbose AI-powered transpiler logs
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)

## Aforderunge

Stellt vor em Beginn von dem Tutorial sicher, dass ir Folgendes installiert hän:

* Qiskit SDK v1.0 oder höher, mit Unterstützig für [Visualisierig](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 oder höher
* Qiskit IBM&reg; Transpiler mit KI-Lokalmodus(`pip install 'qiskit-ibm-transpiler[ai-local-mode]'`)

## Setup

In [2]:
num_circuits_sim = 20
depth_sim = 4
qubit_range_sim = list(range(6, 26))

circuits_sim = [
    # We have only two qubit gates, as those test how well the
    # transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_sim,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_sim)
]

print(
    f"Created {len(circuits_sim)} circuits with qubit counts "
    f"from {qubit_range_sim[0]} to {qubit_range_sim[-1]}"
)
circuits_sim[0].draw(output="mpl", fold=-1)

Created 20 circuits with qubit counts from 6 to 25


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step1-code-1.avif" alt="Output of the previous code cell" />

# Teil I. Qiskit-Muster
Jetzt luege mir uns aa, wie mr de KI-Transpiler-Service mit ere einfache Quanteschaltung unter Verwendung von Qiskit-Mustere verwendet. Dr Schlüssel isch d' Erstellig von emm `PassManager` mit `generate_ai_pass_manager()` anstelle vom Standard `generate_preset_pass_manager()`.

## Schritt 1: Klassischi Iigage uf e Quanteproblem abbilden
In dem Abschnitt teste mir de KI-Transpiler aa dr `efficient_su2`-Schaltung, emm wyt verbreitet hardwareeffizienten Aasatz. Die Schaltung isch besonders relevant für variationelli Quantealgorithme (zum Byschpiel VQE) un Quantum-Machine-Learning-Aufgabe, was sie zu emm ideale Testfall für d' Bewertig von dr Transpilationsleischtig macht.

D' `efficient_su2`-Schaltung besteht us abwechselnde Schichte von Ei-Qubit-Rotatione un verschränkende Gates wie CNOTs. Die Schichte ermögliche e flexible Erkundig vom Quantezustandsraum, während d' Gate-Tiefe überschaubar bliebt. Durch Optimierig von dere Schaltung welle mir d' Gate-Anzahl reduziere, d' Fidelität verbessere un Rausche minimiere. Des macht sie zu emm starke Kandidate zum Teste von dr Effizienz vom KI-Transpilers.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=100, operational=True, simulator=False
)


pm_default_sim = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

In [4]:
results_sim = []

for i, qc in enumerate(circuits_sim):
    n = qubit_range_sim[i]

    qc_default, m_default = transpile_with_metrics(pm_default_sim, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_sim.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_sim)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q               33.0 +/- 12.9          26.4 +/- 8.0      +15.8% +/- 17.6%
Gate Count           522.0 +/- 266.0       560.5 +/- 279.1        -9.0% +/- 9.0%
Time (s)                 0.0 +/- 0.0           0.2 +/- 0.1    -893.6% +/- 362.9%


The summary table shows the mean and standard deviation of each metric across all 20 circuits, along with the average percentage improvement of the AI-powered transpiler over the default. Positive values indicate the AI-powered transpiler produced better results; negative values indicate the default was better.

For this small-scale example, the AI-powered transpiler achieves roughly 16% lower two-qubit depth on average, but at the cost of roughly 9% higher gate count. This highlights a key trade-off when choosing between the two strategies: the AI-powered transpiler prioritizes depth reduction (fewer sequential layers of two-qubit gates), while the default transpiler (SABRE) prioritizes minimizing total gate count (fewer SWAP insertions). Depending on your application, one metric may matter more than the other.

In [5]:
plot_metrics_and_pct(results_sim, "Small-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-1.avif" alt="Output of the previous code cell" />

### KI- un traditionelli Pass-Manager erstelle
Um d' Effektivität vom KI-Transpilers zu bewerte, führe mir zwoi Transpilationsläuf durch. Zerscht transpiliere mir d' Schaltung mit em KI-Transpiler. Derno führe mir e Vergleich durch, indem mir dieselb Schaltung ohni de KI-Transpiler mit traditionelle Methode transpiliere. Beidni Transpilationsprozesse verwende dieselb Coupling-Map vom gwählte Backend un s' Optimierungslevel wird uf 3 gsetzt, für e fairen Vergleich.

Beidni Methode spiegle de Standardaasatz zur Erstellig von `PassManager`-Instanze zur Transpilation von Schaltunge in Qiskit wider.

In [6]:
# Use the 10-qubit circuit (index where qubits == 10)
idx_10q = qubit_range_sim.index(10)

qc_10q = circuits_sim[idx_10q]
qc_default_10q, _ = transpile_with_metrics(pm_default_sim, qc_10q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_10q, _ = transpile_with_metrics(pm_ai, qc_10q)

tqc_methods = {
    "Default": qc_default_10q,
    "AI": qc_ai_10q,
}

print(
    f"Default: depth {qc_default_10q.depth()}, gates {qc_default_10q.size()}"
)
print(f"AI:      depth {qc_ai_10q.depth()}, gates {qc_ai_10q.size()}")

Default: depth 84, gates 280
AI:      depth 91, gates 343


In [7]:
# Build a simple depolarizing noise model
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ["sx", "x", "rz"],  # ~0.1% per 1Q gate
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx", "ecr"],  # ~1% per 2Q gate
)

aer_sim = AerSimulator(noise_model=noise_model)

shots = 10000
survival_probs = {}

for method, tqc in tqc_methods.items():
    mirror = build_mirror_circuit(tqc, simulate=True)

    sampler = SamplerV2(mode=aer_sim)
    job = sampler.run([mirror], shots=shots)
    counts = job.result()[0].data.meas.get_counts()

    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots
    survival_probs[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  "
        f"({counts.get(all_zeros, 0)}/{shots})"
    )

Default   P(|00...0>) = 0.8460  (8460/10000)
AI        P(|00...0>) = 0.8121  (8121/10000)


We ran both mirror circuits through the Aer simulator with a simple depolarizing noise model. The survival probability, defined as the fraction of shots that return the all-zeros bitstring, quantifies how much noise each transpilation strategy introduces.

### Step 4: Post-process and return result in desired classical format

We extract the probability of measuring the all-zeros bitstring from both runs. A higher survival probability indicates better fidelity, meaning the transpilation introduced less noise. The plot below shows the complement, 1 - P(|0...0>), so that a lower bar indicates better fidelity and small differences in error are easier to see.

In [8]:
# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs = {method: 1 - p for method, p in survival_probs.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs.keys(),
    error_probs.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title("Mirror Circuit Error (10-qubit, Aer Simulator)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step4-code-0.avif" alt="Output of the previous code cell" />

In dem Test vergleiche mir d' Leischtig vom KI-Transpiler un von dr Standard-Transpilationsmethode aa dr efficient_su2-Schaltung. Dr KI-Transpiler erreicht e merklich flacheri Schaltungstiefe bi ähnlicher Gate-Anzahl.

- **Schaltungstiefe:** Dr KI-Transpiler produziert e Schaltung mit geringerer Zwei-Qubit-Tiefe. Des isch zu erwarte, wil d' KI-Passes druf trainiert sin, d' Tiefe zu optimiere, indem sie Qubit-Interaktionsmuster lerne un Hardware-Konnektivität effektiver als regelbasierti Heuristike usnutze.

- **Gate-Anzahl:** D' Gesamt-Gate-Anzahl bliebt zwische de zwoi Methode ähnlich. Des entspricht de Erwartunge, wil d' Standard-SABRE-basierti Transpilation explizit d' Swap-Anzahl minimiert, wo de Gate-Overhead dominiert. Dr KI-Transpiler priorisiert stattdesse d' Gesamttiefe un ka gelegentlich e paar zusätzlichi Gates für e kürzere Usführungspfad iitusche.

- **Transpilationszyt:** Dr KI-Transpiler bruucht meh Zyt als d' Standardmethode. Des liegt aa de zusätzliche Rechekoscht für s' Ufrufe von glernde Modelle während vom Routing un dr Synthese. Im Gegensatz dazu isch dr SABRE-basierti Transpiler jetzt nach Neufassung un Optimierig in Rust deutlich schneller un bietet hocheffizients heuristisches Routing im grosse Maßstab.

's isch wichtig z'beachte, dass die Ergebnis nur uf ere Schaltung basiere. Um e umfassendes Verständnis davon z'chriege, wie sich dr KI-Transpiler im Vergleich zu traditionelle Methode verhält, isch es nötig, e Vielzahl von Schaltunge z'teste. D' Leischtig von QTS ka je nach Art von dr zu optimierende Schaltung stark variiere. Für e breiteren Vergleich verwise mir uf d' obige Benchmarks oder besueche de [Blog.](https://www.ibm.com/quantum/blog/qiskit-performance)

## Schritt 3: Usführig mit Qiskit Primitives {#schritt-3-ausführung-mit-qiskit-primitives}
Wil sich des Tutorial uf Transpilation konzentriert, werre kei Experimente uf em Quantegerät usgeführt. S' Ziel isch es, d' Optimierunge us Schritt 2 z'nutze, um e transpilierti Schaltung mit reduzierter Tiefe oder Gate-Anzahl z'chriege.

## Schritt 4: Nachbearbeitung un Rückgab vom Ergebnis im gwünscht klassische Format {#schritt-4-nachbearbeitung-und-rückgabe-des-ergebnisses-im-gewünschten-klassischen-format}
Wil's kei Usführig für des Notebook git, git's kei Ergebnis zur Nachbearbeitung.

# Teil II. Analyse un Benchmarking von de transpilierte Schaltunge
In dem Abschnitt zeige mir, wie mr d' transpilierti Schaltung analysiert un detaillierter mit dr Originalversion vergleicht. Mir konzentriere uns uf Metrike wie Schaltungstiefe, Gate-Anzahl un Transpilationszyt, um d' Effektivität von dr Optimierig z'bewerte. Zusätzlich diskutiere mir, wie d' Ergebnis über verschiedeni Schaltungstyp hii variiere chöi, un biete Einblick in d' breiteiri Leischtig vom Transpiler über verschiedeni Szenarios hii.

In [9]:
# -------------------------Step 1-------------------------
num_circuits_hw = 25
depth_hw = 8
qubit_range_hw = list(range(26, 51))

circuits_hw = [
    # We have only two qubit gates, as those test how well the
    # transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_hw,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_hw)
]

print(
    f"Created {len(circuits_hw)} circuits with qubit counts "
    f"from {qubit_range_hw[0]} to {qubit_range_hw[-1]}"
)

Created 25 circuits with qubit counts from 26 to 50


In [10]:
# -------------------------Step 2-------------------------
pm_default = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

results_hw = []

for i, qc in enumerate(circuits_hw):
    n = qubit_range_hw[i]

    qc_default, m_default = transpile_with_metrics(pm_default, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_hw.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_hw)

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q              217.4 +/- 50.4        191.0 +/- 35.6      +10.9% +/- 10.7%
Gate Count         4513.3 +/- 1394.3     5227.1 +/- 1536.4       -16.4% +/- 5.8%
Time (s)                 0.1 +/- 0.0           3.5 +/- 1.5   -3588.2% +/- 643.6%


In [11]:
plot_metrics_and_pct(results_hw, "Large-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-1.avif" alt="Output of the previous code cell" />

In [12]:
# -------------------------Step 3-------------------------
# Build mirror circuits from the 26-qubit case
idx_26q = qubit_range_hw.index(26)

qc_26q = circuits_hw[idx_26q]
qc_default_26q, _ = transpile_with_metrics(pm_default, qc_26q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_26q, _ = transpile_with_metrics(pm_ai, qc_26q)

mirror_default_hw = build_mirror_circuit(qc_default_26q, simulate=False)
mirror_ai_hw = build_mirror_circuit(qc_ai_26q, simulate=False)

# Re-transpile to basis gates (the inverse can introduce gates like sxdg)
pm_basis = generate_preset_pass_manager(
    optimization_level=0,
    backend=backend,
)
mirror_default_hw = pm_basis.run(mirror_default_hw)
mirror_ai_hw = pm_basis.run(mirror_ai_hw)

print(
    f"Mirror circuit (Default): depth {mirror_default_hw.depth()}, "
    f"gates {mirror_default_hw.size()}"
)
print(
    f"Mirror circuit (AI):      depth {mirror_ai_hw.depth()}, "
    f"gates {mirror_ai_hw.size()}"
)

# Submit to real hardware
sampler_hw = SamplerV2(mode=backend)
sampler_hw.options.environment.job_tags = ["TUT_AITI"]

shots_hw = 500000
job_hw = sampler_hw.run([mirror_default_hw, mirror_ai_hw], shots=shots_hw)
print(f"Job submitted: {job_hw.job_id()}")

Mirror circuit (Default): depth 1577, gates 9672
Mirror circuit (AI):      depth 1235, gates 11092
Job submitted: d8gt7vm6983c73dqbg0g


In [13]:
# -------------------------Step 4-------------------------
result_hw = job_hw.result()

survival_probs_hw = {}
for i, method in enumerate(["Default", "AI"]):
    counts = result_hw[i].data.meas.get_counts()
    mirror = [mirror_default_hw, mirror_ai_hw][i]
    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots_hw
    survival_probs_hw[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  "
        f"({counts.get(all_zeros, 0)}/{shots_hw})"
    )

# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs_hw = {method: 1 - p for method, p in survival_probs_hw.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs_hw.keys(),
    error_probs_hw.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title(f"Mirror Circuit Error (26-qubit, {backend.name})")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

Default   P(|00...0>) = 0.0005  (239/500000)
AI        P(|00...0>) = 0.0050  (2516/500000)


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step4-1.avif" alt="Output of the previous code cell" />

### Analysis of results

The large-scale results reinforce the trends observed in the small-scale example, now at a more demanding scale.

**Two-qubit depth:** The AI-powered transpiler continues to deliver noticeably lower two-qubit depth across the full range of circuit sizes. Depth optimization is one of the primary objectives the AI routing model is trained on, and the advantage is more pronounced at larger qubit counts where the routing problem becomes harder for heuristic methods.

**Gate count:** The default transpiler (SABRE) consistently produces circuits with fewer gates across all circuit sizes in this range. SABRE's heuristic is specifically designed to minimize gate count, and at this scale the advantage is clear and uniform.

**Transpilation time:** The gap in transpilation time widens at larger scales. SABRE remains nearly constant, while the AI-powered transpiler's runtime grows more steeply. Despite this, the AI-powered transpiler runtime remains practical for most workflows.

**Mirror circuit fidelity:** Both methods produce survival probabilities well under 1% at this scale, leaving little usable signal. With total gate counts around 10,000 and two-qubit depths exceeding 1,000, the depolarizing noise accumulated across the mirror circuit overwhelms most of the signal. This highlights a key limitation of the mirror circuit approach: while it is simple and requires no classical simulation, it does not scale well to large or deep circuits, where both methods are pushed close to the noise floor and the small surviving signal is dominated by accumulated error.

While these results underscore the AI-powered transpiler's effectiveness, it is important to note its limitations. The AI synthesis method is currently only available for certain coupling maps, which may restrict its broader applicability. This constraint should be considered when evaluating its usage in different scenarios.

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Transpilation optimizations with SABRE](/docs/tutorials/transpilation-optimizations-with-sabre)
- [Compilation methods for Hamiltonian simulation circuits](/docs/tutorials/compilation-methods-for-hamiltonian-simulation-circuits)

</Admonition>